# Face attendance experiment: detection + ArcFace + PAD

This standalone Colab/Kaggle notebook detects faces, enrolls identities with ArcFace, checks each detected face with MiniFASNet presentation-attack detection (PAD), and exports both all decisions and accepted attendance rows to CSV. **It intentionally does not use ByteTrack, the app's Euler/blink liveness heuristic, its web server, or its database.** It is for model evaluation and camera testing, not a production attendance service.

Use your own consenting test participants. Keep live, printed-photo, and screen-replay examples in separate folders. A small successful test is not a security guarantee; evaluate separately captured, held-out samples from the actual deployment camera.

## 1. Runtime and dependencies

In Colab or Kaggle, first select a GPU accelerator if desired. Run the install cell, then restart the kernel if the notebook platform requests it. For CPU, install `onnxruntime`; for a CUDA-capable runtime, use `onnxruntime-gpu` instead (do not install both). ONNX Runtime providers reported below are authoritative: GPU is used only if `CUDAExecutionProvider` is actually available. InsightFace may download the `buffalo_l` detection/recognition pack on first run.

In [ ]:
%pip install -q "numpy<2" opencv-python-headless==4.10.0.84 insightface==0.7.3
# Choose exactly ONE provider package; restart the runtime after switching:
# CPU runtime:  %pip install -q onnxruntime==1.20.1
# GPU runtime:  %pip install -q onnxruntime-gpu==1.20.1
# Do not install both. The imports cell reports whether CUDAExecutionProvider is really available.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64
import csv
import hashlib
import importlib.util
import multiprocessing as mp
import os
import sys
import tempfile
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from urllib.request import urlopen

import cv2
import numpy as np
import onnxruntime as ort

available = ort.get_available_providers()
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in available else ['CPUExecutionProvider']
USE_GPU = 'CUDAExecutionProvider' in available
print('OpenCV:', cv2.__version__)
print('ONNX Runtime:', ort.__version__)
print('Available providers:', available)
print('Selected providers:', providers)
print('GPU inference enabled:', USE_GPU)

## 2. Mount Drive / choose data folders

Upload or copy your test images into Google Drive, then mount it in Colab. Kaggle notebooks can read a dataset attached under `/kaggle/input/`. Use this directory layout:

```text
facial_pad_test/
  enrollment/
    employee_001/  (several genuine enrollment face photos)
    employee_002/
  evaluation/
    live/          (genuine camera captures)
    print/         (camera images of printed photos held up)
    screen/        (camera images of a phone/monitor replay)
```

Each enrollment folder name becomes the employee ID/name. Use multiple clear, consented enrollment images per person. Do not use the same capture session for both tuning and held-out reporting.

In [ ]:
IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/facial_pad_test')
elif Path('/kaggle/input').exists():
    # Change this to the dataset slug attached to your Kaggle notebook.
    DATA_ROOT = Path('/kaggle/input/facial-pad-test')
else:
    DATA_ROOT = Path('./facial_pad_test')

ENROLLMENT_DIR = DATA_ROOT / 'enrollment'
EVALUATION_DIR = DATA_ROOT / 'evaluation'
OUTPUT_DIR = (Path('/kaggle/working/pad_attendance_output') if Path('/kaggle/input').exists()
              else DATA_ROOT / 'results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECOGNITION_THRESHOLD = 0.30  # project starting value; calibrate on your own identities/camera
PAD_LIVE_THRESHOLD = 0.50      # starting point only; validate against held-out attacks
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
print('Data root:', DATA_ROOT)
print('Enrollment folder exists:', ENROLLMENT_DIR.is_dir())
print('Evaluation folder exists:', EVALUATION_DIR.is_dir())

## 3. Load face detector, ArcFace, and PAD

InsightFace `buffalo_l` supplies SCRFD face detection/landmarks and ArcFace recognition, as used by the project's InsightFace backend. MiniFASNet is a separate PAD classifier. The two ONNX PAD weights below are pinned to one conversion commit and checked by SHA-256. Preprocessing follows the original MiniFASNet demo: BGR crops, 2.7× and 4× context, 80×80, CHW float values in the 0–255 range without normalization. PAD class indices are 0=paper, 1=real, 2=screen.

In [ ]:
from insightface.app import FaceAnalysis

MODEL_COMMIT = '584d4421d7ac42c59e640796f46e886b0095367a'
PAD_MODELS = {
    '2.7_80x80_MiniFASNetV2.onnx': ('0CBE5CAEC95C31DE9D2EF845CB85407D76AECD1B6A2C0E343F7D35306BFBCCB8', 2.7),
    '4_0_0_80x80_MiniFASNetV1SE.onnx': ('A25886A85CDCFA2C4EA23EDB71DE35F250C17827B4CADD253A972B28C80FDF1E', 4.0),
}
PAD_MODEL_DIR = OUTPUT_DIR / 'pad_models'
PAD_MODEL_DIR.mkdir(parents=True, exist_ok=True)

def fetch_verified_pad_models():
    for filename, (expected, _) in PAD_MODELS.items():
        target = PAD_MODEL_DIR / filename
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest().upper() != expected:
            url = f'https://raw.githubusercontent.com/QingHeYang/Silent-Face-Anti-Spoofing-onnx/{MODEL_COMMIT}/onnx/{filename}'
            with urlopen(url, timeout=60) as response:
                data = response.read()
            actual = hashlib.sha256(data).hexdigest().upper()
            if actual != expected:
                raise RuntimeError(f'SHA-256 check failed for {filename}: {actual}')
            target.write_bytes(data)
        print('Verified PAD model:', target.name)

fetch_verified_pad_models()
PAD_SESSIONS = []
for filename, (expected, scale) in PAD_MODELS.items():
    session = ort.InferenceSession(str(PAD_MODEL_DIR / filename), providers=providers)
    PAD_SESSIONS.append((scale, session, session.get_inputs()[0].name, session.get_outputs()[0].name))
    print(filename, 'active providers:', session.get_providers())

FACE_APP = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition'], providers=providers)
FACE_APP.prepare(ctx_id=0 if USE_GPU else -1, det_size=(640, 640))
print('Face detector and ArcFace loaded. GPU requested:', USE_GPU)

## 4. Recognition and PAD functions

For each image, the notebook detects faces, calculates the best enrolled-identity cosine match, and runs PAD on each face box. An attendance decision is accepted only if the identity similarity meets `RECOGNITION_THRESHOLD` **and** the PAD result is class `real` with probability at least `PAD_LIVE_THRESHOLD`. No ByteTrack or temporal track cache is used here. The optional threaded mode parallelizes PAD inference across multiple faces in one frame; one-face frames stay sequential.

In [ ]:
CLASS_NAMES = ('paper', 'real', 'screen')

def pad_crop(frame_bgr, bbox_xyxy, context_scale):
    height, width = frame_bgr.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    x1, x2 = np.clip([x1, x2], 0, width - 1)
    y1, y2 = np.clip([y1, y2], 0, height - 1)
    box_w, box_h = x2 - x1, y2 - y1
    if box_w <= 0 or box_h <= 0:
        raise ValueError('Face box must have positive width and height')
    scale = min(context_scale, (width - 1) / box_w, (height - 1) / box_h)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half_w, half_h = box_w * scale / 2, box_h * scale / 2
    left, top = max(0, int(np.floor(cx - half_w))), max(0, int(np.floor(cy - half_h)))
    right, bottom = min(width, int(np.ceil(cx + half_w)) + 1), min(height, int(np.ceil(cy + half_h)) + 1)
    crop = frame_bgr[top:bottom, left:right]
    if crop.size == 0:
        raise ValueError('Face box produced an empty PAD crop')
    resized = cv2.resize(crop, (80, 80), interpolation=cv2.INTER_LINEAR)
    return np.expand_dims(np.transpose(resized.astype(np.float32), (2, 0, 1)), axis=0)

def predict_pad(frame_bgr, bbox_xyxy):
    fused = np.zeros(3, dtype=np.float64)
    for scale, session, input_name, output_name in PAD_SESSIONS:
        logits = np.asarray(session.run([output_name], {input_name: pad_crop(frame_bgr, bbox_xyxy, scale)})[0]).reshape(-1).astype(np.float64)
        if logits.size != 3 or not np.isfinite(logits).all():
            raise RuntimeError('PAD model returned invalid scores')
        exp_logits = np.exp(logits - logits.max())
        fused += exp_logits / exp_logits.sum()
    fused /= len(PAD_SESSIONS)
    label = CLASS_NAMES[int(np.argmax(fused))]
    live_score = float(fused[1])
    return {'label': label, 'live_score': live_score, 'scores': fused,
            'is_live': label == 'real' and live_score >= PAD_LIVE_THRESHOLD}

def normalized_embedding(face):
    embedding = np.asarray(face.normed_embedding, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(embedding))
    if not np.isfinite(norm) or norm <= 1e-12:
        raise ValueError('ArcFace produced an invalid embedding')
    return embedding / norm

def largest_face(frame_bgr):
    faces = FACE_APP.get(frame_bgr)
    if not faces:
        return None
    return max(faces, key=lambda face: max(0, face.bbox[2] - face.bbox[0]) * max(0, face.bbox[3] - face.bbox[1]))

def build_employee_gallery(enrollment_dir):
    gallery = {}
    if not Path(enrollment_dir).is_dir():
        raise FileNotFoundError(f'Enrollment directory not found: {enrollment_dir}')
    for employee_dir in sorted(Path(enrollment_dir).iterdir()):
        if not employee_dir.is_dir():
            continue
        embeddings = []
        for image_path in sorted(employee_dir.rglob('*')):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            image = cv2.imread(str(image_path))
            if image is None:
                print('Skipping unreadable enrollment image:', image_path)
                continue
            face = largest_face(image)
            if face is None:
                print('No enrollment face detected:', image_path)
                continue
            embeddings.append(normalized_embedding(face))
        if embeddings:
            centroid = np.mean(np.stack(embeddings), axis=0)
            gallery[employee_dir.name] = centroid / np.linalg.norm(centroid)
            print(f'Enrolled {employee_dir.name}: {len(embeddings)} image(s)')
        else:
            print('No usable enrollment images for:', employee_dir.name)
    if not gallery:
        raise RuntimeError('No employee embeddings were enrolled')
    return gallery

def match_identity(face, gallery):
    query = normalized_embedding(face)
    matches = [(employee_id, float(np.dot(query, reference))) for employee_id, reference in gallery.items()]
    employee_id, similarity = max(matches, key=lambda item: item[1])
    if similarity < RECOGNITION_THRESHOLD:
        return None, similarity
    return employee_id, similarity

def process_image(frame_bgr, gallery, source, true_label='unknown', pad_execution='sequential', max_workers=4):
    started = time.perf_counter()
    faces = FACE_APP.get(frame_bgr)
    rows = []
    if not faces:
        return [{'timestamp': datetime.now(timezone.utc).isoformat(), 'source': str(source),
                 'true_label': true_label, 'face_index': -1, 'employee_id': '',
                 'identity_score': 0.0, 'pad_label': 'no_face', 'pad_live_score': 0.0,
                 'pad_passed': False,
                 'attended': False, 'reason': 'no_face_detected',
                 'processing_ms': (time.perf_counter() - started) * 1000}]
    if pad_execution not in {'sequential', 'threaded'}:
        raise ValueError("pad_execution must be 'sequential' or 'threaded'")
    if pad_execution == 'threaded' and len(faces) > 1:
        worker_count = min(max_workers, len(faces))
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            pad_results = list(executor.map(
                lambda face: predict_pad(frame_bgr, face.bbox), faces
            ))
    else:
        pad_results = [predict_pad(frame_bgr, face.bbox) for face in faces]
    for index, (face, pad) in enumerate(zip(faces, pad_results)):
        employee_id, identity_score = match_identity(face, gallery)
        attended = employee_id is not None and pad['is_live']
        reasons = []
        if employee_id is None:
            reasons.append('identity_not_matched')
        if not pad['is_live']:
            reasons.append('pad_rejected')
        rows.append({
            'timestamp': datetime.now(timezone.utc).isoformat(), 'source': str(source),
            'true_label': true_label, 'face_index': index,
            'employee_id': employee_id or '', 'identity_score': identity_score,
            'pad_label': pad['label'], 'pad_live_score': pad['live_score'],
            'pad_passed': pad['is_live'],
            'paper_score': float(pad['scores'][0]), 'screen_score': float(pad['scores'][2]),
            'attended': attended, 'reason': 'accepted' if attended else ';'.join(reasons),
            'processing_ms': (time.perf_counter() - started) * 1000,
        })
    return rows

print('Detection + ArcFace + PAD helpers are ready.')

## 5. Enroll identities and evaluate labeled examples

Enroll from the Drive `enrollment/` directory. Put only genuine enrollment photos there. Then place camera-captured test images in `evaluation/live`, `evaluation/print`, and `evaluation/screen`. The evaluation records every detected face and also records a rejection when no face is found. Do not treat enrollment photos themselves as attack test samples.

In [ ]:
GALLERY = build_employee_gallery(ENROLLMENT_DIR)
all_rows = []
for true_label in ('live', 'print', 'screen'):
    class_dir = EVALUATION_DIR / true_label
    if not class_dir.is_dir():
        print('Evaluation class folder missing:', class_dir)
        continue
    for image_path in sorted(class_dir.rglob('*')):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        image = cv2.imread(str(image_path))
        if image is None:
            print('Skipping unreadable evaluation image:', image_path)
            continue
        all_rows.extend(process_image(image, GALLERY, image_path, true_label))

if not all_rows:
    print('No evaluation images found yet; add images under evaluation/live, evaluation/print, and evaluation/screen.')
else:
    print('Evaluated face decisions:', len(all_rows))
    for row in all_rows[:20]:
        print(row)

## 6. Threading and multiprocessing experiment

**Threads:** the optional mode above shares the already-loaded ONNX sessions and runs PAD for different detected faces concurrently. It is most useful when one frame has multiple faces; a one-face frame intentionally falls back to sequential inference. On GPU, concurrent calls may contend for the same device, so compare timings and verify the session's active providers instead of assuming threads improve performance.

**Processes:** each process has a separate Python memory space. Do not send or fork the loaded InsightFace/ONNX sessions into workers, especially after CUDA initialization; that can duplicate model memory, exhaust VRAM, or hang. The process-pool example below measures only CPU image decoding/resizing from file paths. It is a safe way to learn the process-pool overhead and is **not** a multi-process GPU inference benchmark.

In [ ]:
# Compare sequential and threaded end-to-end decisions on a small, bounded sample.
# Detection/ArcFace remain single-call; only per-face PAD inference is threaded.
benchmark_paths = []
for label in ('live', 'print', 'screen'):
    folder = EVALUATION_DIR / label
    if folder.is_dir():
        benchmark_paths.extend(
            (path, label) for path in sorted(folder.rglob('*'))
            if path.suffix.lower() in IMAGE_EXTENSIONS
        )
benchmark_paths = benchmark_paths[:12]

if not benchmark_paths:
    print('Add labeled live/print/screen images to run the inference benchmark.')
else:
    benchmark_images = []
    for path, label in benchmark_paths:
        image = cv2.imread(str(path))
        if image is not None:
            benchmark_images.append((image, path, label))
    comparison = {}
    for mode in ('sequential', 'threaded'):
        durations = []
        decisions = []
        for image, path, label in benchmark_images:
            started = time.perf_counter()
            rows = process_image(image, GALLERY, path, label, pad_execution=mode)
            durations.append((time.perf_counter() - started) * 1000)
            decisions.append([
                (row['face_index'], row['employee_id'], row['pad_label'],
                 row['pad_passed'], row['attended']) for row in rows
            ])
        comparison[mode] = decisions
        print(f'{mode}: {len(durations)} images; mean={np.mean(durations):.2f} ms/image; '
              f'median={np.median(durations):.2f} ms/image')
        if mode == 'threaded':
            multi_face_images = sum(len(image_decisions) > 1 for image_decisions in decisions)
            print(f'  multi-face images with concurrent PAD work: {multi_face_images}/{len(decisions)}')
    print('Decision outputs identical:', comparison['sequential'] == comparison['threaded'])
    print('PAD active providers:', [session.get_providers() for _, session, _, _ in PAD_SESSIONS])

# Multiprocessing demonstration: CPU-only file decode/resize; no model objects enter workers.
worker_source = '''\
import cv2
def resize_image_for_benchmark(image_path, size):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f'Could not decode image: {image_path}')
    resized = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
    return image_path, tuple(resized.shape)
'''
if len(benchmark_paths) >= 2:
    with tempfile.TemporaryDirectory(prefix='pad_cpu_workers_') as worker_dir:
        worker_file = Path(worker_dir) / 'pad_cpu_benchmark_worker.py'
        worker_file.write_text(worker_source, encoding='utf-8')
        sys.path.insert(0, worker_dir)
        try:
            if __name__ == '__main__':
                mp.freeze_support()
                from pad_cpu_benchmark_worker import resize_image_for_benchmark
                arguments = [(str(path), 256) for path, _ in benchmark_paths]
                started = time.perf_counter()
                serial_results = [resize_image_for_benchmark(*item) for item in arguments]
                serial_ms = (time.perf_counter() - started) * 1000
                started = time.perf_counter()
                with ProcessPoolExecutor(max_workers=min(2, os.cpu_count() or 1),
                                         mp_context=mp.get_context('spawn')) as executor:
                    process_results = list(executor.map(resize_image_for_benchmark,
                                                        [item[0] for item in arguments],
                                                        [item[1] for item in arguments]))
                process_ms = (time.perf_counter() - started) * 1000
                print(f'CPU preprocessing — sequential: {serial_ms:.2f} ms; '
                      f'process pool (startup included): {process_ms:.2f} ms')
                print('Preprocessing outputs identical:', serial_results == process_results)
            else:
                print('Process-pool demo skipped outside the notebook main process.')
        finally:
            sys.path.remove(worker_dir)
else:
    print('Add at least two labeled images to run the CPU process-pool demo.')

In [ ]:
def report_metrics(rows):
    usable = [row for row in rows if row['true_label'] in {'live', 'print', 'screen'}]
    bona_fide = [row for row in usable if row['true_label'] == 'live']
    attacks = [row for row in usable if row['true_label'] in {'print', 'screen'}]
    pad_false_rejects = sum(not row['pad_passed'] for row in bona_fide)
    pad_false_accepts = sum(bool(row['pad_passed']) for row in attacks)
    pad_frr = pad_false_rejects / len(bona_fide) if bona_fide else float('nan')
    pad_far = pad_false_accepts / len(attacks) if attacks else float('nan')
    pad_balanced_accuracy = ((1 - pad_frr) + (1 - pad_far)) / 2 if bona_fide and attacks else float('nan')
    attendance_false_rejects = sum(not row['attended'] for row in bona_fide)
    attendance_false_accepts = sum(bool(row['attended']) for row in attacks)
    attendance_frr = attendance_false_rejects / len(bona_fide) if bona_fide else float('nan')
    attendance_far = attendance_false_accepts / len(attacks) if attacks else float('nan')
    attendance_balanced_accuracy = ((1 - attendance_frr) + (1 - attendance_far)) / 2 if bona_fide and attacks else float('nan')
    print(f'Bona-fide decisions: {len(bona_fide)}; attack decisions: {len(attacks)}')
    print(f'PAD only — false accepts: {pad_false_accepts}; FAR: {pad_far:.3f}; false rejects: {pad_false_rejects}; FRR: {pad_frr:.3f}; balanced accuracy: {pad_balanced_accuracy:.3f}')
    print(f'Combined attendance — false accepts: {attendance_false_accepts}; FAR: {attendance_far:.3f}; false rejects: {attendance_false_rejects}; FRR: {attendance_frr:.3f}; balanced accuracy: {attendance_balanced_accuracy:.3f}')
    for label in ('live', 'print', 'screen'):
        subset = [row for row in usable if row['true_label'] == label]
        print(label, {'accepted': sum(bool(row['attended']) for row in subset),
                      'rejected': sum(not row['attended'] for row in subset)})
    if rows:
        print(f"Mean processing time: {np.mean([row['processing_ms'] for row in rows]):.2f} ms/image")

def save_csv(rows, path):
    if not rows:
        print('No rows to save:', path)
        return
    with open(path, 'w', newline='', encoding='utf-8') as output:
        writer = csv.DictWriter(output, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print('Saved:', path)

if all_rows:
    report_metrics(all_rows)
    attendance_rows = [row for row in all_rows if row['attended']]
    save_csv(all_rows, OUTPUT_DIR / 'all_decisions.csv')
    save_csv(attendance_rows, OUTPUT_DIR / 'attendance.csv')

## 7. Direct camera capture (Colab)

The following cell requests one still image from the browser camera in Colab, then runs the same detector + ArcFace + PAD decision and appends the result to CSV. Grant camera permission when prompted. Kaggle may not expose browser camera APIs; if it does not, capture with your device and put the files in the mounted Drive evaluation folders instead.

In [ ]:
def capture_browser_photo():
    if not IN_COLAB:
        raise RuntimeError('Direct browser camera capture is provided for Google Colab; use Drive images on Kaggle.')
    from IPython.display import Javascript, display
    from google.colab.output import eval_js
    javascript = Javascript('''async function capturePhoto() {
      const video = document.createElement('video');
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream; await video.play();
      await new Promise(resolve => setTimeout(resolve, 1200));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(track => track.stop());
      return canvas.toDataURL('image/jpeg', 0.92);
    }''')
    display(javascript)
    data_url = eval_js('capturePhoto()')
    encoded = data_url.split(',', 1)[1]
    image_bytes = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8)
    frame = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
    if frame is None:
        raise RuntimeError('Could not decode the captured camera image')
    return frame

# Run this cell for one genuine participant, then try again with a print or screen held in front of the camera.
camera_rows = []
camera_frame = capture_browser_photo()
camera_rows.extend(process_image(camera_frame, GALLERY, 'browser_camera', 'unknown'))
for row in camera_rows:
    print(row)
save_csv(camera_rows, OUTPUT_DIR / 'camera_decisions.csv')
save_csv([row for row in camera_rows if row['attended']], OUTPUT_DIR / 'camera_attendance.csv')

## Interpreting this experiment and privacy

This notebook uses InsightFace SCRFD + ArcFace and a two-scale MiniFASNet PAD model. It omits ByteTrack, embedding caches, the app's Euler/blink heuristic, database writes, and the web dashboard. It offers optional face-level PAD threading and a CPU-only process-pool preprocessing demonstration; neither is a claim of improved production throughput. PAD output is a model score, not proof of liveness. Tune thresholds using a calibration set, freeze them, then report false-accept/false-reject behavior on separately captured held-out people, sessions, cameras, lighting, print types, and screen devices. Do not make security claims from a handful of images.

Images and embeddings are biometric data. Keep them in a private Drive/Kaggle dataset, obtain consent, restrict access, and delete temporary runtime files when finished. Do not put private face images into a public notebook or repository. Direct camera capture is available only where the notebook frontend grants camera access; Drive image folders are the fallback. Model sources: [InsightFace](https://github.com/deepinsight/insightface) and the Apache-2.0 [pinned MiniFASNet ONNX conversion](https://github.com/QingHeYang/Silent-Face-Anti-Spoofing-onnx/tree/584d4421d7ac42c59e640796f46e886b0095367a), based on [Minivision Silent-Face-Anti-Spoofing](https://github.com/minivision-ai/Silent-Face-Anti-Spoofing).